# 🏆 مشروع توقع كأس العالم 2026
## المرحلة الثالثة: تدريب النماذج (محدّث بالكامل)

### ترتيب التشغيل الصح:
```
01_data_collection      → جمع البيانات
02_feature_engineering  → بناء الـ Features
04_live_update_engine   → تحديث ELO + الفرق المُقصاة ⬅️ شغّله الأول!
03_model_training       → هنا ← تدريب النموذج على أحدث بيانات
```
> ⚠️ **مهم:** شغّل `04_live_update_engine` قبل الـ notebook ده


## 📦 1. استيراد المكتبات

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
import joblib
from datetime import datetime
from scipy.stats import poisson as sp_poisson
warnings.filterwarnings("ignore")

from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import PoissonRegressor
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import accuracy_score, classification_report
import xgboost as xgb

os.makedirs("models", exist_ok=True)
print("✅ المكتبات جاهزة")
print(f"   تاريخ التشغيل: {datetime.now().strftime('%Y-%m-%d %H:%M')}")


✅ المكتبات جاهزة
   تاريخ التشغيل: 2026-07-08 13:35


## 📥 2. تحميل البيانات (من 01 + 02 + 04)

In [2]:
# ── normalize_team (لازم يكون أول حاجة) ─────────────────
TEAM_NAME_MAP = {
    "Holland"              : "Netherlands",
    "Korea Republic"       : "South Korea",
    "IR Iran"              : "Iran",
    "USA"                  : "United States",
    "Côte d'Ivoire"        : "Ivory Coast",
    "Czechia"              : "Czech Republic",
    "Bosnia-Herzegovina"   : "Bosnia and Herzegovina",
    "Bosnia & Herzegovina" : "Bosnia and Herzegovina",
    "UAE"                  : "United Arab Emirates",
    "FYR Macedonia"        : "North Macedonia",
    "Cabo Verde"           : "Cape Verde",
    "DR Congo"             : "DR Congo",
    "Congo DR"             : "DR Congo",
}
def normalize_team(name):
    if pd.isna(name): return name
    return TEAM_NAME_MAP.get(str(name).strip(), str(name).strip())

# ── تحميل Training Dataset من المرحلة الثانية ────────────
training_df   = pd.read_csv("data/processed/training_dataset.csv", parse_dates=["match_date"])
teams_df      = pd.read_csv("data/raw/teams.csv")
squad_summary = pd.read_csv("data/processed/squad_summary.csv")
team_ratings  = pd.read_csv("data/processed/team_ratings_fc26.csv")
fifa_rank_df  = pd.read_csv("data/processed/fifa_rank_latest.csv")

for df in [teams_df, squad_summary, team_ratings, fifa_rank_df]:
    for col in df.columns:
        if "team" in col.lower() or "country" in col.lower():
            df[col] = df[col].apply(normalize_team)

# ── تحميل ELO — Live لو موجود، قديم لو لأ ────────────────
elo_path = (
    "data/processed/elo_live_updated.csv"
    if os.path.exists("data/processed/elo_live_updated.csv")
    else "data/processed/elo_current.csv"
)
elo_current = pd.read_csv(elo_path)
elo_current["team"] = elo_current["team"].apply(normalize_team)
if "elo_live" in elo_current.columns:
    elo_current = elo_current.rename(columns={"elo_live": "elo"})

# ── تحميل الفرق المُقصاة من الـ 04 ───────────────────────
eliminated_path = "data/processed/eliminated_teams.txt"
if os.path.exists(eliminated_path):
    with open(eliminated_path) as f:
        ELIMINATED_TEAMS = set(f.read().strip().split("\n"))
else:
    ELIMINATED_TEAMS = set()
    print("⚠️  ملف الفرق المُقصاة مش موجود — شغّل 04 الأول!")

# ── تحميل البيانات الأخرى ────────────────────────────────
results_played   = pd.read_csv("data/processed/results_clean.csv",    parse_dates=["date"])
results_official = pd.read_csv("data/processed/results_official.csv", parse_dates=["date"])
shootouts        = pd.read_csv("data/processed/shootouts_clean.csv",  parse_dates=["date"])
shootout_stats   = pd.read_csv("data/processed/shootout_stats.csv")

for col in ["home_team","away_team"]:
    results_played[col]   = results_played[col].apply(normalize_team)
    results_official[col] = results_official[col].apply(normalize_team)
    shootouts[col]        = shootouts[col].apply(normalize_team)
shootouts["winner"]       = shootouts["winner"].apply(normalize_team)
shootout_stats["team"]    = shootout_stats["team"].apply(normalize_team)

print(f"✅ البيانات جاهزة:")
print(f"   Training Dataset : {len(training_df):,} مباراة")
print(f"   ELO محمّل من     : {elo_path.split('/')[-1]}")
print(f"   فرق مُقصاة       : {len(ELIMINATED_TEAMS)}")
if ELIMINATED_TEAMS:
    for t in sorted(ELIMINATED_TEAMS):
        print(f"     ❌ {t}")


✅ البيانات جاهزة:
   Training Dataset : 1,018 مباراة
   ELO محمّل من     : elo_live_updated.csv
   فرق مُقصاة       : 31
     ❌ Chile
     ❌ Costa Rica
     ❌ Cuba
     ❌ Curaçao
     ❌ Czech Republic
     ❌ Germany
     ❌ Haiti
     ❌ Honduras
     ❌ Indonesia
     ❌ Iran
     ❌ Iraq
     ❌ Jamaica
     ❌ Japan
     ❌ Jordan
     ❌ Netherlands
     ❌ New Zealand
     ❌ Panama
     ❌ Peru
     ❌ Poland
     ❌ Qatar
     ❌ Saudi Arabia
     ❌ Scotland
     ❌ Serbia
     ❌ South Africa
     ❌ South Korea
     ❌ Tunisia
     ❌ Turkey
     ❌ Ukraine
     ❌ Uruguay
     ❌ Uzbekistan
     ❌ Venezuela


## 🔧 3. حل مشكلة ELO Dominance — تحويل ELO لاحتمالية
> بدل الفرق الخام (300 نقطة) → احتمالية (88%) — بيمنع ELO يطغى على باقي الـ features


In [3]:
def elo_to_win_prob(elo_diff, home_advantage=50, neutral=True):
    """تحويل فرق ELO لاحتمالية فوز بمعادلة ELO الأصلية"""
    adv = 0 if neutral else home_advantage
    return 1 / (1 + 10 ** (-(elo_diff + adv) / 400))

print("📊 تأثير فرق ELO على احتمالية الفوز:\n")
print(f"{'فرق ELO':>10} | {'احتمالية الفوز':>18} | المعنى")
print("-" * 55)
for diff, meaning in [(-300,"مستحيل تقريباً"),(-150,"ضعيف جداً"),
                       (-50,"أضعف قليلاً"),(0,"متكافئان"),
                       (50,"أقوى قليلاً"),(150,"أقوى بوضوح"),
                       (300,"تفوق كبير — لكن مش مستحيل!")]:
    prob = elo_to_win_prob(diff)
    bar  = "█" * int(prob * 20)
    print(f"{diff:>+10} | {prob*100:>7.1f}%  {bar:<20} | {meaning}")


📊 تأثير فرق ELO على احتمالية الفوز:

   فرق ELO |     احتمالية الفوز | المعنى
-------------------------------------------------------
      -300 |    15.1%  ███                  | مستحيل تقريباً
      -150 |    29.7%  █████                | ضعيف جداً
       -50 |    42.9%  ████████             | أضعف قليلاً
        +0 |    50.0%  ██████████           | متكافئان
       +50 |    57.1%  ███████████          | أقوى قليلاً
      +150 |    70.3%  ██████████████       | أقوى بوضوح
      +300 |    84.9%  ████████████████     | تفوق كبير — لكن مش مستحيل!


## ⚖️ 4. بناء الـ Features المتوازنة

In [4]:
def build_balanced_features(df):
    """إضافة features جديدة لمنع ELO Dominance"""
    df = df.copy()
    df["elo_win_prob"]       = df.apply(
        lambda r: elo_to_win_prob(r["elo_diff"], neutral=bool(r.get("neutral_venue",True))), axis=1)
    df["elo_diff_capped"]    = np.clip(df["elo_diff"], -200, 200)
    df["upset_potential"]    = np.where(
        (df["elo_diff"] < -50) & (df["form10_win_rate_diff"] > 0.15), 1, 0)
    df["ko_upset_risk"]      = df["is_knockout"] * (1 - df["elo_win_prob"])
    df["form_elo_interaction"]= df["form10_win_rate_diff"] * (1 - df["elo_win_prob"])
    return df

training_balanced = build_balanced_features(training_df)

print("✅ Features المتوازنة جاهزة")
print(f"   Features جديدة مضافة:")
for f in ["elo_win_prob","elo_diff_capped","upset_potential","ko_upset_risk","form_elo_interaction"]:
    print(f"   + {f}")
print(f"\n📊 Upset Potential في التاريخ:")
print(training_balanced["upset_potential"].value_counts().to_string())


✅ Features المتوازنة جاهزة
   Features جديدة مضافة:
   + elo_win_prob
   + elo_diff_capped
   + upset_potential
   + ko_upset_risk
   + form_elo_interaction

📊 Upset Potential في التاريخ:
upset_potential
0    975
1     43


## 🎯 5. الـ Features النهائية + RobustScaler

In [5]:
FEATURE_COLS = [
    # ELO متوازن ✅
    "elo_win_prob", "elo_diff_capped", "elo_official_diff",
    # FIFA Ranking
    "fifa_rank_diff",
    # الفورم الحديثة
    "form5_win_rate_diff", "form10_win_rate_diff", "form10_gd_diff", "form10_gf_diff",
    # قوة اللاعبين ⭐
    "avg_overall_diff", "max_overall_diff", "market_value_diff",
    # المواجهات المباشرة ⭐
    "h2h_win_rate_a", "h2h_wc_win_rate_a",
    # ركلات الترجيح ⭐
    "shootout_win_rate_diff",
    # عوامل المباراة ⭐
    "is_knockout", "neutral_venue", "avg_caps_diff",
    # Strength Score ⭐
    "strength_score_diff",
    # Upset Features الجديدة ⭐⭐
    "upset_potential", "ko_upset_risk", "form_elo_interaction",
]

X = training_balanced[FEATURE_COLS].fillna(0)
y = training_balanced["target"]

scaler   = RobustScaler()
X_scaled = scaler.fit_transform(X)

print(f"✅ {len(FEATURE_COLS)} feature جاهزة للتدريب")


✅ 21 feature جاهزة للتدريب


## ✂️ 6. تقسيم زمني — قبل/بعد 2018

In [6]:
split_year = 2018
train_mask = training_balanced["match_date"].dt.year < split_year
test_mask  = training_balanced["match_date"].dt.year >= split_year

X_train_scaled = X_scaled[train_mask]
X_test_scaled  = X_scaled[test_mask]
y_train        = y[train_mask]
y_test         = y[test_mask]

print(f"✅ التقسيم الزمني:")
print(f"   Training : {train_mask.sum():,} مباراة (1930–{split_year-1})")
print(f"   Testing  : {test_mask.sum():,}  مباراة ({split_year}–2026)")


✅ التقسيم الزمني:
   Training : 836 مباراة (1930–2017)
   Testing  : 182  مباراة (2018–2026)


## 🚀 7. تدريب XGBoost المتوازن

In [7]:
xgb_model = xgb.XGBClassifier(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    min_child_weight=3, gamma=0.1,
    use_label_encoder=False, eval_metric="mlogloss",
    random_state=42, n_jobs=-1,
)

cv      = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(xgb_model, X_train_scaled, y_train, cv=cv, scoring="accuracy")

print(f"📊 XGBoost Cross-Validation (5-fold):")
print(f"   Accuracy: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")
print(f"   Scores  : {[f'{s:.3f}' for s in cv_scores]}")

xgb_model.fit(X_train_scaled, y_train)

y_pred   = xgb_model.predict(X_test_scaled)
test_acc = accuracy_score(y_test, y_pred)
print(f"\n✅ Test Accuracy: {test_acc:.3f} ({test_acc*100:.1f}%)")
print(f"\n{classification_report(y_test, y_pred, target_names=['فوز team_b','تعادل','فوز team_a'])}")

joblib.dump(xgb_model, "models/xgb_model_balanced.pkl")
print("💾 محفوظ: models/xgb_model_balanced.pkl")


📊 XGBoost Cross-Validation (5-fold):
   Accuracy: 0.469 ± 0.041
   Scores  : ['0.399', '0.503', '0.461', '0.467', '0.515']

✅ Test Accuracy: 0.538 (53.8%)

              precision    recall  f1-score   support

  فوز team_b       0.52      0.50      0.51        58
       تعادل       0.20      0.07      0.11        42
  فوز team_a       0.59      0.80      0.68        82

    accuracy                           0.54       182
   macro avg       0.44      0.46      0.43       182
weighted avg       0.48      0.54      0.49       182

💾 محفوظ: models/xgb_model_balanced.pkl


## 📊 8. Feature Importance — هل ELO اتوازن؟

In [8]:
importance_df = pd.DataFrame({
    "feature"   : FEATURE_COLS,
    "importance": xgb_model.feature_importances_
}).sort_values("importance", ascending=False).reset_index(drop=True)
importance_df["importance_pct"] = (importance_df["importance"] * 100).round(2)
importance_df["rank"] = importance_df.index + 1

print("🏆 ترتيب الـ Features:\n")
display(importance_df[["rank","feature","importance_pct"]].rename(
    columns={"rank":"#","feature":"الـ Feature","importance_pct":"التأثير %"}))

elo_total   = importance_df[importance_df["feature"].isin(
    ["elo_win_prob","elo_diff_capped","elo_official_diff"])]["importance_pct"].sum()
other_total = 100 - elo_total
print(f"\n📊 ELO features مجتمعة : {elo_total:.1f}%")
print(f"   باقي الـ features   : {other_total:.1f}%")
print("✅ ELO متوازن!" if elo_total < 40 else "⚠️  ELO لسه مؤثر أكتر من اللازم")


🏆 ترتيب الـ Features:



,#,الـ Feature,التأثير %
0,1,avg_overall_diff,8.13
1,2,neutral_venue,7.53
2,3,fifa_rank_diff,5.73
3,4,form5_win_rate_diff,5.68
4,5,elo_diff_capped,5.46
5,6,strength_score_diff,5.27
6,7,elo_official_diff,5.25
7,8,avg_caps_diff,5.11
8,9,market_value_diff,5.03
9,10,elo_win_prob,5.01



📊 ELO features مجتمعة : 15.7%
   باقي الـ features   : 84.3%
✅ ELO متوازن!


## ⚽ 9. Poisson Regression — توقع عدد الأهداف

In [9]:
X_goals        = training_balanced[FEATURE_COLS].fillna(0)
X_goals_scaled = scaler.transform(X_goals)
y_home         = training_balanced["home_score"]
y_away         = training_balanced["away_score"]

poisson_home = PoissonRegressor(alpha=0.1, max_iter=500)
poisson_away = PoissonRegressor(alpha=0.1, max_iter=500)
poisson_home.fit(X_goals_scaled, y_home)
poisson_away.fit(X_goals_scaled, y_away)

home_mae = np.abs(poisson_home.predict(X_goals_scaled) - y_home).mean()
away_mae = np.abs(poisson_away.predict(X_goals_scaled) - y_away).mean()

print(f"✅ Poisson Models:")
print(f"   Home Goals MAE: {home_mae:.3f}")
print(f"   Away Goals MAE: {away_mae:.3f}")
print(f"   (أقل من 1 هدف = كويس ✅)")

joblib.dump(poisson_home,  "models/poisson_home.pkl")
joblib.dump(poisson_away,  "models/poisson_away.pkl")
joblib.dump(scaler,        "models/scaler.pkl")
joblib.dump(FEATURE_COLS,  "models/feature_cols.pkl")
print("\n💾 كل النماذج محفوظة في models/")


✅ Poisson Models:
   Home Goals MAE: 1.068
   Away Goals MAE: 0.895
   (أقل من 1 هدف = كويس ✅)

💾 كل النماذج محفوظة في models/


## 🔧 10. الدوال المساعدة للتوقع

In [10]:
def calculate_recent_form(results_df, team, before_date=None, n_matches=10, official_only=True):
    df = results_df[results_df["tournament_weight"]>1].copy() if official_only else results_df.copy()
    if before_date: df = df[df["date"] < pd.Timestamp(before_date)]
    home_g = df[df["home_team"]==team].copy()
    home_g["scored"]=home_g["home_score"]; home_g["conceded"]=home_g["away_score"]
    home_g["points"]=home_g["result"].map({"home_win":3,"draw":1,"away_win":0})
    away_g = df[df["away_team"]==team].copy()
    away_g["scored"]=away_g["away_score"]; away_g["conceded"]=away_g["home_score"]
    away_g["points"]=away_g["result"].map({"away_win":3,"draw":1,"home_win":0})
    all_g  = pd.concat([home_g,away_g]).sort_values("date",ascending=False).head(n_matches)
    if len(all_g)==0:
        return {f"form{n_matches}_points":0, f"form{n_matches}_gf":0,
                f"form{n_matches}_ga":0,     f"form{n_matches}_gd":0,
                f"form{n_matches}_win_rate":0,f"form{n_matches}_matches":0}
    return {f"form{n_matches}_points"  : int(all_g["points"].sum()),
            f"form{n_matches}_gf"      : round(all_g["scored"].mean(),2),
            f"form{n_matches}_ga"      : round(all_g["conceded"].mean(),2),
            f"form{n_matches}_gd"      : round((all_g["scored"]-all_g["conceded"]).mean(),2),
            f"form{n_matches}_win_rate": round((all_g["points"]==3).mean(),3),
            f"form{n_matches}_matches" : len(all_g)}

def get_h2h(results_df, team_a, team_b, wc_only=False, before_date=None):
    df = results_df.copy()
    if wc_only:    df = df[df["tournament"]=="FIFA World Cup"]
    if before_date:df = df[df["date"]<pd.Timestamp(before_date)]
    mask = ((df["home_team"]==team_a)&(df["away_team"]==team_b)) |            ((df["home_team"]==team_b)&(df["away_team"]==team_a))
    h2h  = df[mask].copy()
    if len(h2h)==0:
        return {"h2h_played":0,"h2h_wins_a":0,"h2h_wins_b":0,
                "h2h_draws":0,"h2h_win_rate_a":0.333,"h2h_avg_goals":0}
    wins_a = len(h2h[((h2h["home_team"]==team_a)&(h2h["result"]=="home_win"))|
                     ((h2h["away_team"]==team_a)&(h2h["result"]=="away_win"))])
    wins_b = len(h2h[((h2h["home_team"]==team_b)&(h2h["result"]=="home_win"))|
                     ((h2h["away_team"]==team_b)&(h2h["result"]=="away_win"))])
    draws  = len(h2h[h2h["result"]=="draw"])
    return {"h2h_played":len(h2h),"h2h_wins_a":wins_a,"h2h_wins_b":wins_b,
            "h2h_draws":draws,"h2h_win_rate_a":round(wins_a/len(h2h),3),
            "h2h_avg_goals":round(h2h["total_goals"].mean(),2) if "total_goals" in h2h.columns else 0}

def get_shootout_stats(team):
    row = shootout_stats[shootout_stats["team"]==team]
    if len(row)==0: return {"shootout_played":0,"shootout_won":0,"shootout_win_rate":0.5}
    r = row.iloc[0]
    return {"shootout_played":int(r["shootout_played"]),
            "shootout_won"   :int(r["shootout_won"]),
            "shootout_win_rate":float(r["shootout_win_rate"])}

def build_team_features(team_name):
    features = {"team": team_name}
    elo_row  = elo_current[elo_current["team"]==team_name]
    features["elo"] = float(elo_row["elo"].values[0]) if len(elo_row)>0 else 1500.0
    team_row = teams_df[teams_df["team_name"]==team_name]
    if len(team_row)>0:
        features["fifa_rank"]    = int(team_row["fifa_ranking_pre_tournament"].values[0])
        features["elo_official"] = float(team_row["elo_rating"].values[0])
    else:
        fifa_row = fifa_rank_df[fifa_rank_df["team"]==team_name]
        features["fifa_rank"]    = int(fifa_row["fifa_rank"].values[0]) if len(fifa_row)>0 else 100
        features["elo_official"] = features["elo"]
    f5  = calculate_recent_form(results_played, team_name, n_matches=5)
    f10 = calculate_recent_form(results_played, team_name, n_matches=10)
    features.update(f5); features.update(f10)
    fc_row = team_ratings[team_ratings["team"]==team_name]
    features["avg_overall"] = float(fc_row["avg_overall"].values[0]) if len(fc_row)>0 else 75.0
    features["max_overall"] = float(fc_row["max_overall"].values[0]) if len(fc_row)>0 else 80.0
    sq_row = squad_summary[squad_summary["team"]==team_name]
    features["total_market_value"] = float(sq_row["total_market_value"].values[0]) if len(sq_row)>0 else 0.0
    features["avg_market_value"]   = float(sq_row["avg_market_value"].values[0])   if len(sq_row)>0 else 0.0
    features["avg_caps"]           = float(sq_row["avg_caps"].values[0])            if len(sq_row)>0 else 0.0
    so = get_shootout_stats(team_name)
    features.update(so)
    return features

def build_match_features(team_a, team_b, match_date=None, is_knockout=False, neutral=True):
    fa = build_team_features(team_a)
    fb = build_team_features(team_b)
    h2h_all = get_h2h(results_played, team_a, team_b, before_date=match_date)
    h2h_wc  = get_h2h(results_played, team_a, team_b, wc_only=True, before_date=match_date)
    elo_diff  = fa["elo"] - fb["elo"]
    form_diff = fa["form10_win_rate"] - fb["form10_win_rate"]
    elo_norm_a = fa["elo"]/2200; elo_norm_b = fb["elo"]/2200
    val_norm_a = fa["total_market_value"]/1_500_000_000
    val_norm_b = fb["total_market_value"]/1_500_000_000
    ovr_norm_a = fa["avg_overall"]/90; ovr_norm_b = fb["avg_overall"]/90
    sa = 0.35*elo_norm_a + 0.25*fa["form10_win_rate"] + 0.20*val_norm_a + 0.20*ovr_norm_a
    sb = 0.35*elo_norm_b + 0.25*fb["form10_win_rate"] + 0.20*val_norm_b + 0.20*ovr_norm_b
    mf = {
        "team_a":team_a, "team_b":team_b,
        "elo_diff"           : elo_diff,
        "elo_win_prob"       : elo_to_win_prob(elo_diff, neutral=neutral),
        "elo_diff_capped"    : np.clip(elo_diff, -200, 200),
        "elo_official_diff"  : fa["elo_official"] - fb["elo_official"],
        "fifa_rank_diff"     : fb["fifa_rank"]    - fa["fifa_rank"],
        "form5_win_rate_diff": round(fa["form5_win_rate"]  - fb["form5_win_rate"],  3),
        "form10_win_rate_diff":round(form_diff, 3),
        "form10_gd_diff"     : round(fa["form10_gd"] - fb["form10_gd"], 2),
        "form10_gf_diff"     : round(fa["form10_gf"] - fb["form10_gf"], 2),
        "avg_overall_diff"   : round(fa["avg_overall"] - fb["avg_overall"], 1),
        "max_overall_diff"   : round(fa["max_overall"] - fb["max_overall"], 1),
        "market_value_diff"  : round(fa["total_market_value"] - fb["total_market_value"], 0),
        "avg_market_value_diff":round(fa["avg_market_value"] - fb["avg_market_value"], 0),
        "h2h_played"         : h2h_all["h2h_played"],
        "h2h_win_rate_a"     : h2h_all["h2h_win_rate_a"],
        "h2h_wc_played"      : h2h_wc["h2h_played"],
        "h2h_wc_win_rate_a"  : h2h_wc["h2h_win_rate_a"],
        "shootout_win_rate_a" : fa["shootout_win_rate"],
        "shootout_win_rate_b" : fb["shootout_win_rate"],
        "shootout_win_rate_diff":round(fa["shootout_win_rate"]-fb["shootout_win_rate"],3),
        "avg_caps_diff"      : round(fa["avg_caps"] - fb["avg_caps"], 1),
        "is_knockout"        : int(is_knockout),
        "neutral_venue"      : int(neutral),
        "strength_score_a"   : round(sa, 4),
        "strength_score_b"   : round(sb, 4),
        "strength_score_diff": round(sa - sb, 4),
        "elo_a":fa["elo"], "elo_b":fb["elo"],
        "fifa_rank_a":fa["fifa_rank"], "fifa_rank_b":fb["fifa_rank"],
        "avg_overall_a":fa["avg_overall"], "avg_overall_b":fb["avg_overall"],
        "form10_wr_a":fa["form10_win_rate"], "form10_wr_b":fb["form10_win_rate"],
    }
    mf["upset_potential"]     = int((elo_diff < -50) and (form_diff > 0.15))
    mf["ko_upset_risk"]       = int(is_knockout) * (1 - mf["elo_win_prob"])
    mf["form_elo_interaction"] = form_diff * (1 - mf["elo_win_prob"])
    return mf

print("✅ كل الدوال جاهزة!")


✅ كل الدوال جاهزة!


## 🔮 11. دالة التوقع الشاملة

In [11]:
def predict_match(team_a, team_b, is_knockout=False, neutral=True):
    """
    توقع كامل لمباراة — XGBoost + Poisson Ensemble.
    ⚠️ بيتحقق أوتوماتيك لو الفريق مُقصى من البطولة
    """
    # تحقق من الفرق المُقصاة
    for team in [team_a, team_b]:
        if team in ELIMINATED_TEAMS:
            print(f"⛔ {team} مُقصى من البطولة — لا يمكن التوقع!")
            return None

    mf   = build_match_features(team_a, team_b, is_knockout=is_knockout, neutral=neutral)
    x    = np.array([[mf.get(f,0) for f in FEATURE_COLS]])
    x_sc = scaler.transform(x)

    # XGBoost
    proba              = xgb_model.predict_proba(x_sc)[0]
    p_b_xgb, p_d_xgb, p_a_xgb = proba[0]*100, proba[1]*100, proba[2]*100

    # Poisson
    lam_h = max(0.3, poisson_home.predict(x_sc)[0])
    lam_a = max(0.3, poisson_away.predict(x_sc)[0])
    ph = pd_ = pa = 0
    for h in range(10):
        for a in range(10):
            p = sp_poisson.pmf(h,lam_h) * sp_poisson.pmf(a,lam_a)
            if h>a: ph+=p
            elif h==a: pd_+=p
            else: pa+=p

    # Ensemble
    p_home = round((p_a_xgb + ph*100) / 2, 1)
    p_draw = round((p_d_xgb + pd_*100) / 2, 1)
    p_away = round((p_b_xgb + pa*100)  / 2, 1)

    if p_home >= p_away and p_home >= p_draw:   winner = f"🏆 {team_a}"
    elif p_away >= p_home and p_away >= p_draw:  winner = f"🏆 {team_b}"
    else:                                         winner = "🤝 تعادل"

    stage = "⚡ KNOCKOUT" if is_knockout else "🔵 دور المجموعات"
    print("=" * 58)
    print(f"  {team_a}  🆚  {team_b}")
    print(f"  {stage}")
    print("=" * 58)
    print(f"\n  النتيجة المتوقعة  :  {lam_h:.1f} — {lam_a:.1f}")
    print(f"\n  الاحتمالية (Ensemble):")
    print(f"    فوز {team_a:<22} {p_home:>5.1f}%  {'█'*int(p_home/4)}")
    print(f"    تعادل                      {p_draw:>5.1f}%  {'█'*int(p_draw/4)}")
    print(f"    فوز {team_b:<22} {p_away:>5.1f}%  {'█'*int(p_away/4)}")
    print(f"\n  التوقع النهائي: {winner}")
    if mf["upset_potential"]:
        print(f"  ⚠️  تحذير: upset محتمل!")
    print("=" * 58)

    return {"team_a":team_a,"team_b":team_b,
            "p_home":p_home,"p_draw":p_draw,"p_away":p_away,
            "exp_home":round(lam_h,2),"exp_away":round(lam_a,2),"winner":winner}

# ── جرّب! ────────────────────────────────────────────────
r = predict_match("Spain", "France", is_knockout=True)


  Spain  🆚  France
  ⚡ KNOCKOUT

  النتيجة المتوقعة  :  1.4 — 1.3

  الاحتمالية (Ensemble):
    فوز Spain                   38.2%  █████████
    تعادل                       22.1%  █████
    فوز France                  39.7%  █████████

  التوقع النهائي: 🏆 France


## 🎲 12. Monte Carlo — نسب الفوز بالكأس (المتأهلين فقط)

In [12]:
def simulate_match_once(team_a, team_b):
    mf    = build_match_features(team_a, team_b, is_knockout=True, neutral=True)
    x     = np.array([[mf.get(f,0) for f in FEATURE_COLS]])
    x_sc  = scaler.transform(x)
    lam_h = max(0.3, poisson_home.predict(x_sc)[0])
    lam_a = max(0.3, poisson_away.predict(x_sc)[0])
    gh    = np.random.poisson(lam_h)
    ga    = np.random.poisson(lam_a)
    if gh > ga:    return team_a
    elif ga > gh:  return team_b
    else:
        so_a = get_shootout_stats(team_a)["shootout_win_rate"]
        so_b = get_shootout_stats(team_b)["shootout_win_rate"]
        tot  = so_a + so_b or 1
        return team_a if np.random.random() < so_a/tot else team_b

def run_monte_carlo(teams_list, n=1000):
    import random
    wins = {t:0 for t in teams_list}
    for _ in range(n):
        remaining = teams_list.copy()
        random.shuffle(remaining)
        while len(remaining) > 1:
            nxt = []
            for i in range(0, len(remaining)-1, 2):
                nxt.append(simulate_match_once(remaining[i], remaining[i+1]))
            if len(remaining)%2==1: nxt.append(remaining[-1])
            remaining = nxt
        wins[remaining[0]] += 1
    return pd.DataFrame([
        {"#":i+1,"الفريق":t,"% الفوز بالكأس":round(w/n*100,2)}
        for i,(t,w) in enumerate(sorted(wins.items(),key=lambda x:-x[1]))
    ])

# ── تحميل الفرق المتأهلة من الـ 04 ──────────────────────
alive_path = "data/processed/eliminated_teams.txt"
if os.path.exists(alive_path):
    with open(alive_path) as f:
        eliminated = set(f.read().strip().split("\n"))
    all_wc = set(teams_df["team_name"])
    still_alive = all_wc - eliminated
    print(f"✅ فرق متأهلة فعلياً: {len(still_alive)}")
    print(f"   مُقصاة: {len(eliminated)}")
else:
    # fallback — أقوى 16 فريق بالـ ELO
    still_alive = set(elo_current.sort_values("elo",ascending=False).head(16)["team"])
    print("⚠️  ملف المُقصاة مش موجود — بنستخدم أقوى 16 بالـ ELO")

print(f"\n⏳ جاري تشغيل 2000 محاكاة على {len(still_alive)} فريق...")
mc = run_monte_carlo(list(still_alive), n=1000)
mc.to_csv("data/processed/monte_carlo_results.csv", index=False)

print("\n🏆 نسب الفوز بكأس العالم 2026 (محدّثة):\n")
display(mc.head(16))


✅ فرق متأهلة فعلياً: 29
   مُقصاة: 31

⏳ جاري تشغيل 2000 محاكاة على 29 فريق...

🏆 نسب الفوز بكأس العالم 2026 (محدّثة):



,#,الفريق,% الفوز بالكأس
0,1,France,15.5
1,2,Spain,14.4
2,3,Brazil,9.5
3,4,England,8.8
4,5,Argentina,7.4
5,6,Belgium,6.6
6,7,Colombia,6.3
7,8,Portugal,6.1
8,9,Croatia,5.8
9,10,Sweden,4.0


## ✅ 13. ملخص

In [13]:
print("📁 النماذج المحفوظة في models/:")
for f in sorted(os.listdir("models")):
    size = os.path.getsize(f"models/{f}") // 1024
    print(f"   {f:40s} {size:>5} KB")

print("\n📁 الملفات في data/processed/:")
for f in sorted(os.listdir("data/processed")):
    size = os.path.getsize(f"data/processed/{f}") // 1024
    print(f"   {f:40s} {size:>5} KB")

print(f"\n🚀 جاهزين للمرحلة الخامسة — Dashboard احترافي!")
print(f"   تاريخ التدريب: {datetime.now().strftime('%Y-%m-%d %H:%M')}")


📁 النماذج المحفوظة في models/:
   .ipynb_checkpoints                           0 KB
   feature_cols.pkl                             0 KB
   poisson_away.pkl                             1 KB
   poisson_home.pkl                             1 KB
   scaler.pkl                                   1 KB
   xgb_model.pkl                             1218 KB
   xgb_model_balanced.pkl                    1222 KB

📁 الملفات في data/processed/:
   eliminated_teams.txt                         0 KB
   elo_current.csv                              4 KB
   elo_history.csv                            895 KB
   elo_live_updated.csv                         4 KB
   fifa_rank_latest.csv                         5 KB
   manual_overrides.csv                         0 KB
   monte_carlo_live.csv                         0 KB
   monte_carlo_results.csv                      0 KB
   results_clean.csv                         4765 KB
   results_future.csv                           1 KB
   results_official.csv              